# How to over-approximate a ReLU — in-class demo

**998MG Safe and Verified AI — 2026/27 · Lectures 9–11 · Università degli Studi di Trieste**

Every incomplete verifier does the same thing: it pushes a **set** of inputs through the network,
and at every ReLU it replaces the true graph of $y = \mathrm{relu}(x)$, $x \in [l, u]$, with a
convex shape that contains it. Affine layers are easy; **the ReLU is where precision is lost**.
This notebook computes, on the examples of the slides, what each method does with it.

| method | the ReLU on $[l, u]$, $l < 0 < u$, becomes | bounds come from | slides |
|---|---|---|---|
| **Interval** (IBP) | the box $[l,u] \times [0,u]$ | reading off the box | L09–10, 15–28 |
| **Zonotope** (DeepZ) | a parallelogram, slope $\lambda = u/(u-l)$, one new generator | a closed formula | L09–10, 31–43, 47–50 |
| **Polyhedron** | the triangle (the LP of L08) | one LP per bound | L09–10, 52–57 |
| **CROWN** (≈ DeepPoly) | the chord above, **one** line $y \ge \alpha x$ below | backward substitution to the input | L11, 8–27 |
| **α-CROWN** | the same, with the slopes $\alpha$ optimized | backward substitution + optimization | L11, 35–38 |

Plan:

1. **one ReLU**, four shapes, and their areas (L11, slides 10–11);
2. **the running example** (L09–10, slides 25–27 and 47–48; L11, slides 3 and 14–22): intervals fail on
   the larger box, zonotopes, the LP and CROWN prove the property, and nobody reaches the true minimum;
3. **choosing the lines**: the CROWN example of L11 (slides 30–38), where the lower slope $\alpha$ decides
   the bound, and α-CROWN picks it.

Only NumPy, SciPy and Matplotlib. Everything runs in a few seconds.

In [ ]:
# numpy for the domains, SciPy's linprog for the polyhedron, matplotlib for the pictures
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction
from scipy.optimize import linprog

np.set_printoptions(precision=4, suppress=True)

# one colour per method, the same in every figure
COLOR = {"interval": "#2a78d6", "zonotope": "#eb6834", "polyhedron": "#1baf7a", "linear": "#eda100"}
INK, MUTED = "#1B2B33", "#8A979D"

# numbers as on the slides: a fraction when the value is one (1/2, 5/8, -10/3), otherwise a decimal
def num(v):
    f = Fraction(float(v)).limit_denominator(64)
    return (str(f) if abs(float(f) - v) < 1e-9 else f"{v:.4g}").replace("-", "−")

# a sum of terms as on the slides: zero terms dropped, coefficient 1 omitted, "−" instead of "+ -"
def join_terms(terms):                                              # terms: (coefficient, name or "")
    terms = [(a, n) for a, n in terms if abs(a) > 1e-12] or [(0.0, "")]
    text = ""
    for k, (a, n) in enumerate(terms):
        mag = num(abs(a)) if not n else ("" if abs(abs(a) - 1) < 1e-12 else num(abs(a)) + "·") + n
        sign = ("−" if a < 0 else "") if k == 0 else (" − " if a < 0 else " + ")
        text += sign + mag
    return text

# a linear expression:  a1·x1 + a2·x2 + const
def line(coeffs, names, const=0.0):
    return join_terms(list(zip(coeffs, names)) + [(const, "")])

## 1. One ReLU, four shapes

Take one neuron whose input ranges over $x \in [l, u]$ with $l < 0 < u$ (an **unstable** ReLU; if
$l \ge 0$ or $u \le 0$ the ReLU is linear on $[l, u]$ and every method is exact, L11 slide 9). The true
set of pairs $(x, y)$ is the graph of the ReLU, two segments meeting at the origin. Each method
replaces it with a convex set:

- **interval**: it only keeps $y \in [\mathrm{relu}(l), \mathrm{relu}(u)] = [0, u]$, and forgets that
  $y$ depends on $x$: the box (L09–10, slide 20);
- **zonotope**: the band between two parallel lines of slope $\lambda = u/(u-l)$,
  $\;\lambda x \le y \le \lambda x + u(1-\lambda)$, written as $y = \lambda x + \eta + \eta\,\varepsilon_{new}$
  with $\eta = u(1-\lambda)/2$ (L09–10, slides 38–40). A zonotope is always centrally symmetric;
- **polyhedron**: the smallest convex set, the triangle $\;y \ge 0,\; y \ge x,\; y \le \lambda (x - l)$
  (L09–10, slide 53: the LP of L08);
- **CROWN**: the same upper side as the triangle (the **chord**), but **one** lower line $y \ge \alpha x$,
  $\alpha \in [0, 1]$: $\alpha = 0$ or $\alpha = 1$, whichever gives the smaller area (L11, slides 10–11).

We take $[l, u] = [-1, 2]$, the example of L11, slides 10–11.

In [ ]:
# area of a polygon given by its corners in order (shoelace formula)
def area(corners):
    x, y = np.array(corners).T
    return abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1))) / 2

# the corners of each relaxation of y = relu(x) on [l, u], l < 0 < u
def relaxations(l, u, alpha):
    lam = u / (u - l)                         # slope of the chord and of the parallelogram
    d = u * (1 - lam)                         # height of the parallelogram (= 2·eta)
    return {
        "interval":   [(l, 0), (u, 0), (u, u), (l, u)],
        "zonotope":   [(l, lam * l), (u, lam * u), (u, lam * u + d), (l, lam * l + d)],
        "polyhedron": [(l, 0), (0, 0), (u, u)],
        "linear":     [(l, alpha * l), (u, alpha * u), (u, u), (l, 0)],   # alpha·x <= y <= lam·(x - l)
    }

# CROWN's rule for the lower slope (DeepPoly's area rule): the smaller of the two triangles.
# On a tie (u = -l) the areas are equal; like auto_LiRPA, we take alpha = 0.
def adaptive_alpha(l, u):
    return 1.0 if u > -l else 0.0

def draw_relu_relaxations(l, u):
    alpha = adaptive_alpha(l, u)
    other = area(relaxations(l, u, 1 - alpha)["linear"])
    titles = {"interval": "Interval: box", "zonotope": "Zonotope: parallelogram",
              "polyhedron": "Polyhedron: triangle",
              "linear": f"CROWN: α = {alpha:g}  (α = {1 - alpha:g}: area {num(other)})"}
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.8), sharex=True, sharey=True)
    for ax, (name, corners) in zip(axes, relaxations(l, u, alpha).items()):
        ax.fill(*np.array(corners).T, color=COLOR[name], alpha=0.25, lw=0)
        ax.plot(*np.array(corners + corners[:1]).T, color=COLOR[name], lw=1.5)
        if name == "linear":                                     # the lower line CROWN does not pick
            ax.plot([l, u], [(1 - alpha) * l, (1 - alpha) * u], color=MUTED, lw=1, ls="--")
        ax.plot([l, 0, u], [0, 0, u], color=INK, lw=2.5)         # the true ReLU
        ax.set_title(f"{titles[name]}\narea {num(area(corners))}", fontsize=10)
        ax.axhline(0, color=MUTED, lw=0.5); ax.axvline(0, color=MUTED, lw=0.5)
        ax.set_xlabel("x")
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
    axes[0].set_ylabel("y = relu(x)")
    plt.tight_layout(); plt.show()

L, U = -1.0, 2.0
draw_relu_relaxations(L, U)

The areas follow a fixed pattern:

| relaxation | area | here, $[-1, 2]$ |
|---|---|---|
| box | $u(u-l)$ | 6 |
| parallelogram | $-l\,u$: **twice** the triangle (L09–10, slide 43) | 2 |
| CROWN, $\alpha = 0$ | $u(u-l)/2$ | 3 |
| CROWN, $\alpha = 1$ | $-l(u-l)/2$ | 3/2 |
| triangle | $-l\,u/2$: the smallest convex set, no single-neuron relaxation does better | 1 |

Here $u = 2 > 1 = -l$, so CROWN picks $\alpha = 1$: $y \ge x$. The shape it picks is never larger than
the parallelogram; the other one can be (here 3 against 2).

### The shape matters, not just the range of $y$

Project each shape on the $y$ axis: that is the bound on $y$ alone.

In [ ]:
lam = U / (U - L)
for name, (lo, hi) in {"interval":   (0, U),
                       "zonotope":   (lam * L, U),         # the lower line lam·x reaches lam·l < 0
                       "polyhedron": (0, U),
                       "CROWN":      (adaptive_alpha(L, U) * L, U)}.items():
    print(f"{name:10}  y in [{num(lo + 0.0):>5}, {num(hi)}]")

The zonotope and CROWN give the **worst** bounds on $y$ alone: they let $y$ be negative, which a ReLU
never is (on L11, slide 18, the interval of a ReLU output is then raised to 0: also sound). Yet in part 2
both prove what intervals cannot. The reason is that they keep the **link
between $y$ and $x$** (a line in $x$, or shared noise symbols): when $y$ is combined with other neurons
later, terms cancel. Intervals keep only the projection, and lose the link.

## 2. The running example

The network of L09–10 (slides 25–27, 47–48) and L11 (slides 3, 14–22), after Singh et al. (DeepPoly,
POPL 2019):

```
x3 = x1 + x2        x5 = relu(x3)        x7 = x5 + x6 − 0.5      x9  = relu(x7)      x11 = −x9 + x10 + 3
x4 = x1 − x2        x6 = relu(x4)        x8 = x5 − x6            x10 = relu(x8)      x12 = x10
```

**Property:** $x_{11} > x_{12}$. Since $x_{11} - x_{12} = 3 - x_9$, it says $x_9 < 3$. Two input boxes:
the small one of slide 26, $x_1 \in [-1, 0]$, $x_2 \in [-1, 1]$, and the larger one of slide 27 and of
L11, $x_1, x_2 \in [-1, 1]$.

A network is a list of layers $(W, b)$, with a ReLU after every layer but the last. `SPEC` is the
linear function of the outputs we bound: here $x_{11} - x_{12}$.

In [ ]:
NET = [(np.array([[1., 1.], [1., -1.]]), np.array([0., 0.])),        # x3, x4
       (np.array([[1., 1.], [1., -1.]]), np.array([-0.5, 0.])),      # x7, x8   (from x5, x6)
       (np.array([[-1., 1.], [0., 1.]]), np.array([3., 0.]))]        # x11, x12 (from x9, x10)
NAMES = [("x3", "x4"), ("x7", "x8"), ("x11", "x12")]                 # the neurons, layer by layer
RELU_NAMES = [("x5", "x6"), ("x9", "x10")]                           # the outputs of the ReLUs
BOX_SMALL = np.array([[-1., 0.], [-1., 1.]])                         # slide 26: x1 in [-1, 0], x2 in [-1, 1]
BOX       = np.array([[-1., 1.], [-1., 1.]])                         # slide 27 and L11: [-1, 1]^2
SPEC = np.array([[1., -1.]])                                         # we bound SPEC · output = x11 − x12

def relu(x):
    return np.maximum(x, 0)

# the concrete network, on a batch of inputs (one per row)
def run(net, x):
    for k, (W, b) in enumerate(net):
        x = x @ W.T + b
        if k < len(net) - 1:
            x = relu(x)
    return x

# the true range of SPEC · output, by brute force on a fine grid (fine for 2 inputs, impossible in general)
def true_range(net, box, spec, n=801):
    grid = np.meshgrid(*[np.linspace(lo, hi, n) for lo, hi in box])
    out = run(net, np.stack([g.ravel() for g in grid], axis=1)) @ spec.T
    return out.min(), out.max()

for label, box in [("small box (slide 26)", BOX_SMALL), ("larger box (slide 27)", BOX)]:
    lo, hi = true_range(NET, box, SPEC)
    print(f"{label}:  true range of x11 − x12 = [{num(lo)}, {num(hi)}]")

The property holds on both boxes. Can the methods prove it?

### Intervals

Affine: each bound takes every input at its worst end (positive weight → same end, negative
weight → the other end). ReLU: monotone, so $[\mathrm{relu}(l), \mathrm{relu}(u)]$.

In [ ]:
def interval(net, box, verbose=True, names=None):
    lo, hi = box[:, 0], box[:, 1]
    for k, (W, b) in enumerate(net):
        Wp, Wn = np.maximum(W, 0), np.minimum(W, 0)
        lo, hi = Wp @ lo + Wn @ hi + b, Wp @ hi + Wn @ lo + b       # affine: the worst corner, row by row
        if verbose:
            print("  " + ",  ".join(f"{names[k][j] if names else f'layer {k + 1}.{j + 1}'} in "
                                    f"[{num(lo[j])}, {num(hi[j])}]" for j in range(len(lo))))
        if k < len(net) - 1:
            lo, hi = relu(lo), relu(hi)                             # ReLU: monotone
    return lo, hi

for label, box in [("small box (slide 26)", BOX_SMALL), ("larger box (slide 27)", BOX)]:
    print(label)
    lo, hi = interval(NET, box, names=NAMES)
    verdict = "proved" if lo[0] > hi[1] else "unknown"
    print(f"  x11 > x12?  l11 = {num(lo[0])} > u12 = {num(hi[1])}  ->  {verdict}\n")

On the small box intervals prove the property: $x_{11} \ge 1.5 > 1 \ge x_{12}$ (slide 26). On the larger
box they fail: $-0.5 > 2$ is false (slide 27). Two things go wrong:

- the outputs are bounded **separately** and then compared, so $x_{10}$, which appears in both and cancels
  in $x_{11} - x_{12} = 3 - x_9$, is counted twice;
- even on the difference, $3 - x_9 \ge 3 - 3.5 = -0.5$: intervals give $x_7 \le 3.5$, because they
  do not know that $x_5$ and $x_6$ cannot both be 2 (L11, slide 16).

Bounding the difference directly is easy: append `SPEC` to the last layer. Then the network has one
output, $x_{11} - x_{12}$.

In [ ]:
# the last layer followed by SPEC is still one affine layer: (SPEC·W, SPEC·b)
def with_spec(net, spec):
    W, b = net[-1]
    return net[:-1] + [(spec @ W, spec @ b)]

lo, hi = interval(with_spec(NET, SPEC), BOX, verbose=False)
print(f"intervals on the difference, larger box:  x11 − x12 >= {num(lo[0])}   ->  unknown")

### Zonotopes

Every neuron is $c_0 + \sum_i c_i \varepsilon_i$ with $\varepsilon_i \in [-1, 1]$, stored as a centre $c$ and a
matrix $G$ (one row per neuron, one column per $\varepsilon_i$). The larger box is $x_1 = \varepsilon_1$,
$x_2 = \varepsilon_2$. Affine layers are exact; each unstable ReLU becomes the parallelogram and adds one
column (L09–10, slides 38–39). The steps below are those of slide 48.

In [ ]:
# x = c + G·eps, eps in [-1, 1]^m:  the bounds set every eps_i to ±1 following the sign of its coefficient
def zbounds(c, G):
    r = np.abs(G).sum(axis=1)
    return c - r, c + r

# a zonotope row as text:  c0 + c1·ε1 + ...
def zstr(c0, g):
    return join_terms([(c0, "")] + [(gi, f"ε{i + 1}") for i, gi in enumerate(g)])

def zonotope(net, box, verbose=True, names=None, relu_names=None):
    c = box.mean(axis=1)
    G = np.diag((box[:, 1] - box[:, 0]) / 2)                      # one generator per input
    for k, (W, b) in enumerate(net):
        c, G = W @ c + b, W @ G                                     # affine: exact
        if k == len(net) - 1:
            break
        l, u = zbounds(c, G)
        new_columns = []
        for j in range(len(c)):
            name = names[k][j] if names else f"layer {k + 1}.{j + 1}"
            if u[j] <= 0:                                           # stable, inactive: y = 0
                c[j], G[j] = 0, 0
            elif l[j] < 0:                                          # unstable: the parallelogram
                lam = u[j] / (u[j] - l[j])
                eta = u[j] * (1 - lam) / 2
                c[j], G[j] = lam * c[j] + eta, lam * G[j]
                col = np.zeros(len(c)); col[j] = eta                # the new generator, only on neuron j
                new_columns.append(col)
                if verbose:
                    print(f"  relu({name}), {name} in [{num(l[j])}, {num(u[j])}]: unstable, "
                          f"λ = {num(lam)}, η = {num(eta)}, new generator ε{G.shape[1] + len(new_columns)}")
            elif verbose:
                print(f"  relu({name}), {name} in [{num(l[j])}, {num(u[j])}]: stable, active")
        if new_columns:
            G = np.hstack([G, np.array(new_columns).T])
        if verbose:
            for j in range(len(c)):
                lo_j, hi_j = zbounds(c[j:j + 1], G[j:j + 1])
                out = relu_names[k][j] if relu_names else f"relu(layer {k + 1}.{j + 1})"
                print(f"    {out} = {zstr(c[j], G[j])}   in [{num(lo_j[0])}, {num(hi_j[0])}]")
    return c, G

c, G = zonotope(NET, BOX, names=NAMES, relu_names=RELU_NAMES)
lo, hi = zbounds(c, G)
print(f"\nx11 in [{num(lo[0])}, {num(hi[0])}],  x12 in [{num(lo[1])}, {num(hi[1])}]")
# the zonotope subtracts first, then bounds: the shared generators cancel
d_c, d_G = SPEC @ c, SPEC @ G
print(f"x11 − x12 = {zstr(d_c[0], d_G[0])}")
print(f"x11 − x12 >= {num(zbounds(d_c, d_G)[0][0])}   ->  proved")

The numbers of slides 47–48: $\lambda = \eta = 1/2$ for $x_3$ and $x_4$; $x_7 \in [-1.5, 2.5]$ with
$\lambda = 5/8$, $\eta = 15/32$; $x_9 = 25/32 + \tfrac58\varepsilon_1 + \tfrac5{16}\varepsilon_3 + \tfrac5{16}\varepsilon_4 + \tfrac{15}{32}\varepsilon_5$;
and $x_{11} - x_{12} = 3 - x_9 \ge 71/32 - 55/32 = 1/2 > 0$: **proved**.

Two more details are visible above. The ReLU outputs $x_5$, $x_6$, $x_{10}$ get a negative
lower bound, $-1$: the parallelogram goes below 0, and it is still sound. And the upper bound of $x_{11}$,
$5.44$, is **worse** than with intervals, $5$: zonotopes are not more precise on every neuron, they win
because $\varepsilon_2$ cancels in $x_7$ and $x_{10}$ cancels in the difference.

### Polyhedra

The same generators, but each unstable ReLU's new generator is **not** in $[-1, 1]$: it is the output of
the ReLU itself, tied to its input by the three triangle constraints (L09–10, slides 54–56):
$\varepsilon_{new} \ge 0$, $\;\varepsilon_{new} \ge x$, $\;\varepsilon_{new} \le \lambda (x - l)$. The constraints are
collected in $F$; every bound, intermediate ones included, is an LP over $F$ (slide 55).

In [ ]:
# a polyhedron: x = c + G·e, with the generators e subject to F:  A·e <= rhs,  and box bounds on each e_i
def polyhedron(net, box, verbose=True, names=None):
    c = box.mean(axis=1)
    G = np.diag((box[:, 1] - box[:, 0]) / 2)
    gen_bounds = [(-1, 1)] * len(c)                                 # the input generators, as for zonotopes
    A, rhs = [], []

    # min (sense = +1) or max (sense = -1) of  c0 + g·e  subject to F: one LP
    def optimise(c0, g, sense):
        res = linprog(sense * g, A_ub=np.array(A) if A else None, b_ub=rhs if A else None, bounds=gen_bounds)
        return c0 + g @ res.x

    def bounds(c, G):
        return (np.array([optimise(c[j], G[j], +1) for j in range(len(c))]),
                np.array([optimise(c[j], G[j], -1) for j in range(len(c))]))

    n_lp = 0
    for k, (W, b) in enumerate(net):
        c, G = W @ c + b, W @ G                                     # affine: exact, F unchanged
        if k == len(net) - 1:
            break
        l, u = bounds(c, G); n_lp += 2 * len(c)
        for j in range(len(c)):
            name = names[k][j] if names else f"layer {k + 1}.{j + 1}"
            if u[j] <= 0:
                c[j], G[j] = 0, 0
            elif l[j] < 0:                                          # unstable: a new generator e_new = relu(x_j)
                lam = u[j] / (u[j] - l[j])
                G = np.hstack([G, np.zeros((len(c), 1))])
                A = [np.append(a, 0) for a in A]
                gen_bounds = gen_bounds + [(None, None)]
                x_j = G[j].copy()                                   # x_j = c[j] + x_j·e  (before we overwrite row j)
                e_new = np.zeros(G.shape[1]); e_new[-1] = 1
                A += [-e_new,                                       # e_new >= 0
                      x_j - e_new,                                  # e_new >= x_j
                      e_new - lam * x_j]                            # e_new <= lam·(x_j − l)
                rhs += [0, -c[j], lam * (c[j] - l[j])]
                c[j], G[j] = 0, e_new                               # the neuron is now exactly e_new
                if verbose:
                    print(f"  relu({name}), {name} in [{num(l[j])}, {num(u[j])}] (2 LPs): "
                          f"new generator ε{G.shape[1]} with 3 constraints")
    if verbose:
        print(f"  {n_lp} LPs for the intermediate bounds, {len(A)} constraints in F")
    return c, G, optimise

c, G, optimise = polyhedron(NET, BOX, names=NAMES)
d_c, d_G = (SPEC @ c)[0], (SPEC @ G)[0]
print(f"\nx11 − x12 = {zstr(d_c, d_G)}")
print(f"x11 − x12 >= {num(optimise(d_c, d_G, +1))}   ->  proved")

$x_{11} - x_{12} = 3 - \varepsilon_5$, where $\varepsilon_5$ stands for $x_9$, tied to $x_7$ by its triangle. Again
$1/2$: the polyhedron is the most precise of the three domains, and still not exact. We come back to
the reason after CROWN.

### CROWN: two lines per neuron, substituted backwards

Each neuron keeps **one lower and one upper line** in terms of the neurons before it (L11, slide 14). For
an unstable ReLU, the chord above and $\alpha x$ below:

$$\alpha\, x \;\le\; \mathrm{relu}(x) \;\le\; \lambda\,(x - l), \qquad \alpha \in [0, 1].$$

To bound a linear function of the output, say $x_{11} - x_{12}$, we walk **backwards** layer by layer and
replace each neuron by one of its lines (L11, slides 23–25):

- an affine layer is an equality: substitute it;
- a ReLU with a **positive** coefficient, for a lower bound, gets its **lower** line; with a
  **negative** coefficient it gets its **upper** line (a negative coefficient flips the inequality).

At the input we have $x_{11} - x_{12} \ge a^\top x + \text{const}$, and the box gives the number
(concretization, slide 13). The pre-activation bounds $l, u$ of each layer, needed to draw the lines,
are computed the same way, one backward pass per layer (slide 26). DeepPoly keeps the same lines and
substitutes the same way: on ReLU networks it computes the same bounds (slide 27).

In [ ]:
# lower bound of  spec · f(x)  on the box, by back-substitution.
#   pre[k]   = (l, u), the pre-activation bounds of hidden layer k
#   alpha[k] = the lower slopes of its ReLUs
# returns the bound, and the linear function Lam·x + const it comes from
def backsubstitute(net, box, spec, pre, alpha):
    Lam = np.array(spec, dtype=float)                               # coefficients on the current layer
    const = np.zeros(len(Lam))
    for k in range(len(net) - 1, -1, -1):
        W, b = net[k]
        const += Lam @ b                                            # affine layer: substitute it
        Lam = Lam @ W
        if k == 0:
            break
        l, u = pre[k - 1]                                           # the ReLUs feeding layer k
        for j in range(len(l)):
            if l[j] >= 0:                                           # stable active: relu(x) = x
                continue
            if u[j] <= 0:                                           # stable inactive: relu(x) = 0
                Lam[:, j] = 0
                continue
            lam = u[j] / (u[j] - l[j])
            pos = Lam[:, j] >= 0
            # positive coefficient: lower line alpha·x;  negative: upper line lam·x − lam·l
            const += np.where(pos, 0, Lam[:, j] * (-lam * l[j]))
            Lam[:, j] = np.where(pos, Lam[:, j] * alpha[k - 1][j], Lam[:, j] * lam)
    centre, radius = box.mean(axis=1), (box[:, 1] - box[:, 0]) / 2
    return const + Lam @ centre - np.abs(Lam) @ radius, Lam, const  # min over the box of Lam·x + const

# bounds of spec · f(x) with a rule (l, u) -> alpha for every unstable ReLU
def linear_bounds(net, box, spec, rule, verbose=False, names=None, relu_names=None):
    pre = []
    for k in range(len(net) - 1):                                   # pre-activation bounds, layer by layer
        I = np.eye(len(net[k][1]))
        alpha = [rule(l, u) for l, u in pre]
        l = backsubstitute(net[:k + 1], box, I, pre, alpha)[0]
        u = -backsubstitute(net[:k + 1], box, -I, pre, alpha)[0]
        pre.append((l, u))
        if verbose:                                                 # the lines of each ReLU (L11, slide 18)
            for j in range(len(l)):
                name = names[k][j] if names else f"layer {k + 1}.{j + 1}"
                out = relu_names[k][j] if relu_names else f"relu({name})"
                if l[j] < 0 < u[j]:
                    lam, a = u[j] / (u[j] - l[j]), rule(l, u)[j]
                    print(f"  {name} in [{num(l[j])}, {num(u[j])}]:  "
                          f"{line(np.array([a]), [name])} <= {out} <= {line(np.array([lam]), [name], -lam * l[j])}")
                else:
                    print(f"  {name} in [{num(l[j])}, {num(u[j])}]:  stable")
    alpha = [rule(l, u) for l, u in pre]
    lower, Lam, const = backsubstitute(net, box, spec, pre, alpha)
    upper = -backsubstitute(net, box, -spec, pre, alpha)[0]
    return lower, upper, Lam, const

# CROWN's adaptive rule (DeepPoly's area rule), neuron by neuron; a tie gives alpha = 0
adaptive = lambda l, u: np.where(u > -l, 1.0, 0.0)

lower, upper, Lam, const = linear_bounds(NET, BOX, SPEC, adaptive, verbose=True, names=NAMES,
                                         relu_names=RELU_NAMES)
print(f"\nx11 − x12 >= {line(Lam[0], ['x1', 'x2'], const[0])}")
print(f"x11 − x12 >= {num(lower[0])}   ->  proved")

Every line of L11, slide 18: $x_3, x_4 \in [-2, 2]$ are ties, so $\alpha = 0$ and
$0 \le x_5 \le \tfrac12 x_3 + 1$; by back-substitution $x_7 \in [-0.5, 2.5]$ (intervals gave 3.5, slides
16–17), and $x_7 \le x_9 \le \tfrac56 x_7 + \tfrac5{12}$. The final line is the one of slide 21,
$x_{11} - x_{12} \ge 4/3 - \tfrac56 x_1$, with minimum $1/2$ at $x_1 = 1$. No LP at all.

### Where is the gap?

Four methods, one box (L11, slide 22). The true minimum is 1.5.

In [ ]:
# one horizontal bar per method, with the true range behind them
def plot_ranges(results, true_lo, true_hi, xlabel, zero=False):
    fig, ax = plt.subplots(figsize=(9, 0.55 * len(results) + 1.2))
    ax.axvspan(true_lo, true_hi, color=MUTED, alpha=0.2, lw=0)
    ax.text(true_hi, len(results) - 0.35, f" true range [{num(true_lo)}, {num(true_hi)}]",
            color=INK, fontsize=9, va="center")
    if zero:
        ax.axvline(0, color=INK, lw=1, ls=":")
    family = {"interval": "interval", "zonotope": "zonotope", "polyhedron": "polyhedron"}
    for i, (name, (lo, hi)) in enumerate(reversed(list(results.items()))):
        ax.plot([lo, hi], [i, i], color=COLOR[family.get(name.split()[0].rstrip(","), "linear")], lw=6,
                solid_capstyle="round")
        ax.text(lo, i + 0.28, num(lo), ha="center", fontsize=8, color=INK)
        ax.text(hi, i + 0.28, num(hi), ha="center", fontsize=8, color=INK)
    ax.set_yticks(range(len(results)), list(reversed(list(results))))
    ax.set_xlabel(xlabel)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.tick_params(axis="y", length=0)
    plt.tight_layout(); plt.show()

# every method on the difference x11 − x12 (intervals: also bounding the two outputs separately)
lo, hi = interval(NET, BOX, verbose=False)
i_lo, i_hi = interval(with_spec(NET, SPEC), BOX, verbose=False)
c, G = zonotope(with_spec(NET, SPEC), BOX, verbose=False)
c_p, G_p, opt = polyhedron(with_spec(NET, SPEC), BOX, verbose=False)
results = {
    "interval, l11 − u12": (lo[0] - hi[1], hi[0] - lo[1]),
    "interval, on x11 − x12": (i_lo[0], i_hi[0]),
    "zonotope": tuple(v[0] for v in zbounds(c, G)),
    "polyhedron (LP)": (opt(c_p[0], G_p[0], +1), opt(c_p[0], G_p[0], -1)),
    "CROWN": tuple(v[0] for v in linear_bounds(NET, BOX, SPEC, adaptive)[:2]),
}
plot_ranges(results, *true_range(NET, BOX, SPEC), "x11 − x12 on [-1, 1]²", zero=True)

# the culprit: the upper bound of x7 = x5 + x6 − 0.5, before the ReLU x9
first_two = NET[:2]
e7 = np.array([[1., 0.]])
c, G = zonotope(first_two, BOX, verbose=False)
c_p, G_p, opt = polyhedron(first_two, BOX, verbose=False)
print("upper bound of x7:  intervals", num(interval(first_two, BOX, verbose=False)[1][0]),
      "| zonotope", num(zbounds(c, G)[1][0]),
      "| polyhedron", num(opt(c_p[0], G_p[0], -1)),
      "| CROWN", num(linear_bounds(first_two, BOX, e7, adaptive)[1][0]),
      "| true", num(true_range(first_two, BOX, e7)[1]))

Zonotope, polyhedron and CROWN all stop at $1/2$, for the same reason: they all bound $x_7 \le 2.5$, while
really $x_7 \le 1.5$ (L11, slides 16–17 and 22). $x_7 = x_5 + x_6 - 0.5$, and each ReLU is relaxed **on its
own**: no single-neuron relaxation knows that $x_5 = \mathrm{relu}(x_1 + x_2)$ and
$x_6 = \mathrm{relu}(x_1 - x_2)$ cannot both be large.

The upper bounds in the picture tell the rest: intervals on the difference 3, polyhedron 3, CROWN 7/2,
zonotope 63/16. The methods are not ordered (the zonotope was also worse than intervals on $x_{11}$); only
the polyhedron is always at least as good as the others, and it pays one LP per bound.

## 3. Choosing the lines: the CROWN example of L11

$x_1 \in [-1, 2]$, $x_2 \in [-2, 1]$ (L11, slides 30–38, after H. Zhang's tutorial):

```
z1 = x1 − x2        ẑ1 = relu(z1)
z2 = 2·x1 − x2      ẑ2 = relu(z2)        y = ẑ1 − ẑ2
```

Both ReLUs are unstable: $z_1 \in [-2, 4]$, $z_2 \in [-3, 6]$, and $\lambda = 2/3$ for both. Without the
ReLUs, $y = -x_1 \in [-2, 1]$ (L11, slides 4–5).

In [ ]:
NET_B  = [(np.array([[1., -1.], [2., -1.]]), np.array([0., 0.])),
          (np.array([[1., -1.]]), np.array([0.]))]
BOX_B  = np.array([[-1., 2.], [-2., 1.]])
SPEC_B = np.eye(1)                                                  # we bound y itself

lo, hi = true_range(NET_B, BOX_B, SPEC_B)
print(f"true range of y: [{num(lo)}, {num(hi)}]")

Two rules for $\alpha$:

- **same slope**, $\alpha = \lambda$: the lower line parallel to the chord, as on slides 30–31. It is the
  choice of Fast-Lin (Weng et al., 2018), and the zonotope's parallelogram;
- **CROWN** (DeepPoly's area rule): $\alpha = 1$ for both, since $u > -l$ (slide 33).

In [ ]:
parallel = lambda l, u: np.clip(u / np.maximum(u - l, 1e-12), 0, 1)

for name, rule in [("same slope, α = λ", parallel), ("CROWN, α by area", adaptive)]:
    lower, upper, Lam, const = linear_bounds(NET_B, BOX_B, SPEC_B, rule)
    print(f"{name:18}  y >= {line(Lam[0], ['x1', 'x2'], const[0])}"
          f"   ->  y in [{num(lower[0])}, {num(upper[0])}]")

The numbers of slides 31 and 33. Neither rule is better: CROWN has the better **lower** bound ($-3$
against $-10/3$), the same slope the better **upper** bound ($2$ against $3$). Both are sound, since
**every** $\alpha \in [0,1]$ is: $\alpha z$ is below the ReLU on the whole interval (slide 32).

Which $\alpha$ matters for the lower bound? Follow the signs: in $y = \hat z_1 - \hat z_2$, $\hat z_1$ has
coefficient $+1$ (lower line, slope $\alpha_1$) and $\hat z_2$ has $-1$ (upper line, no $\alpha$): the lower
bound depends **only on $\alpha_1$** (slide 36). So we can simply try every value.

In [ ]:
pre_B = [(np.array([-2., -3.]), np.array([4., 6.]))]               # the pre-activation bounds, exact here
alphas = np.linspace(0, 1, 201)
lower_of_a1 = [backsubstitute(NET_B, BOX_B, SPEC_B, pre_B, [np.array([a, 0.])])[0][0] for a in alphas]

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(alphas, lower_of_a1, color=COLOR["linear"], lw=2)
for a, label in [(0, "α₁ = 0"), (0.5, "α₁ = 1/2"), (2 / 3, "α₁ = λ"), (1.0, "CROWN")]:
    v = backsubstitute(NET_B, BOX_B, SPEC_B, pre_B, [np.array([a, 0.])])[0][0]
    ax.plot(a, v, "o", ms=7, color=INK, mec="white", mew=2, zorder=5)
    ax.annotate(f"{label}: {num(v)}", (a, v), textcoords="offset points", xytext=(-6, 8),
                ha="right" if a > 0 else "left", fontsize=9, color=INK)
ax.set_title(f"lower bound of y as a function of α₁   (best: {num(max(lower_of_a1))})", fontsize=10)
ax.set_xlabel("α₁"); ax.margins(y=0.2)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout(); plt.show()

Slide 36: $L(\alpha_1) = 4\alpha_1 - 6$ on $[0, 2/3]$ and $\alpha_1 - 4$ on $[2/3, 1]$; the best slope is
$\alpha_1 = 1$, the CROWN choice. This is **α-CROWN**: treat the slopes as parameters and optimize them,
a separate set for each bound (Xu et al., ICLR 2021, slide 37). Here a grid is enough; in a real network
there are millions of $\alpha$'s, optimized by gradient ascent on the GPU.

### All the methods on this example (slide 38)

In [ ]:
def best_alpha_bounds(net, box, spec, pre):                         # alpha-CROWN by brute force (2 neurons)
    grid = [np.array([a1, a2]) for a1 in alphas[::4] for a2 in alphas[::4]]
    lower = max(backsubstitute(net, box, spec, pre, [a])[0][0] for a in grid)
    upper = min(-backsubstitute(net, box, -spec, pre, [a])[0][0] for a in grid)
    return lower, upper

c, G = zonotope(NET_B, BOX_B, verbose=False)
c_p, G_p, optimise = polyhedron(NET_B, BOX_B, verbose=False)
results = {
    "interval":            tuple(v[0] for v in interval(NET_B, BOX_B, verbose=False)),
    "zonotope":            tuple(v[0] for v in zbounds(c, G)),
    "same slope, α = λ":   linear_bounds(NET_B, BOX_B, SPEC_B, parallel)[:2],
    "CROWN":               linear_bounds(NET_B, BOX_B, SPEC_B, adaptive)[:2],
    "α-CROWN":             best_alpha_bounds(NET_B, BOX_B, SPEC_B, pre_B),
    "polyhedron (LP)":     (optimise(c_p[0], G_p[0], +1), optimise(c_p[0], G_p[0], -1)),
}
results = {k: (float(np.squeeze(lo)), float(np.squeeze(hi))) for k, (lo, hi) in results.items()}
plot_ranges(results, *true_range(NET_B, BOX_B, SPEC_B), "bound on y")

What to read in the picture:

- **intervals** are by far the worst: $[-6, 4]$ against the true $[-2, 1]$;
- the **zonotope** and the same-slope rule give the same bounds, $[-10/3, 2]$. Not a coincidence: the
  parallelogram *is* the band between the parallel lines $\lambda z$ and $\lambda (z - l)$ (they agree on
  the running example too, $[1/2, 63/16]$);
- **CROWN** and **$\alpha = \lambda$** are not comparable: each wins one side;
- **α-CROWN** reaches the **polyhedron** (the triangle LP of L08): with fixed intermediate bounds, the
  best slopes give the LP bound, without solving an LP;
- nobody reaches the true range: every method relaxes each ReLU **separately**. The rest of the gap is the
  *convex barrier* (Salman et al., NeurIPS 2019); closing it needs splits: branch and bound, next.

## Summary

| | ReLU becomes | cost of a bound | relational | §2: $x_{11} - x_{12} \ge$ | §3: $y \in$ |
|---|---|---|---|---|---|
| interval | box | read off | no | $-2.5$ ($-0.5$ on the difference) ? | $[-6, 4]$ |
| zonotope | parallelogram, +1 generator | closed formula | yes, symmetric | $0.5$ ✓ | $[-10/3, 2]$ |
| polyhedron | triangle, +1 generator | one LP | yes | $0.5$ ✓ | $[-3, 2]$ |
| CROWN (≈ DeepPoly) | chord + $y \ge \alpha x$, $\alpha$ by area | back-substitution | yes | $0.5$ ✓ | $[-3, 3]$ |
| α-CROWN | the same, $\alpha$ optimized | back-substitution + gradient steps | yes | $0.5$ ✓ (auto_LiRPA) | $[-3, 2]$ |
| true | | | | $1.5$ | $[-2, 1]$ |

